In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))


In [3]:
import gradio as gr

In [4]:
from Day4LLMCalling import Llms,mSeries

In [5]:
def laugh(text):
    print('laugh initiated...')
    return f'Laughing like a {text.upper()}' 
    

In [6]:
# laugh('wahahaha')

In [7]:
# gr.Interface(fn=Llms.callModel,inputs='textbox', outputs='textbox', flagging_mode='never').launch(inbrowser=True, auth=('edd','howdy'))

In [8]:
def wrapLlm(message,source):
    yield from Llms.callModelGenerator(message,source=source)

In [9]:
message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Markdown(label="Response:")
message_model_selection = gr.Dropdown(choices=['ollama','gemini','openRouter'], label='Choose model')

view = gr.Interface(
    fn=wrapLlm,
    title="Shout", 
    inputs=[message_input,message_model_selection], 
    outputs=[message_output], 
    examples=[["hello there. Introduce yourself.",'openRouter'], ["howdy",'gemini']], 
    flagging_mode="never",
    theme='soft'
    )
view.launch()

c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [10]:
# Llms.callModel('hi there',stream=True,source='openRouter',model='openrouter/free')

In [11]:
def wrapLlm(message,source,model,temperature):
    # if not source:
    #     source = 'openRouter'
    # if not model:
    #     model = 'gpt-oss:20b'
    yield from Llms.callModelGenerator(message,source=source, model=model, temperature=temperature)

In [15]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    # with gr.Group():
        with gr.Row():
            with gr.Column(scale=1,elem_id='col1', variant='panel'):
                chat_list = gr.State([]) 
                new_chat_btn = gr.Button('New',variant='huggingface',size='sm')

                @gr.render(inputs=chat_list)
                def render_chats(chats):
                    for chat in chats:
                        gr.Label(chat)

                def create_new_chat(chats):
                    chats.append(f'chat{len(chats)+1}')
                    return chats

                new_chat_btn.click(fn = create_new_chat, inputs=[chat_list], outputs=[chat_list])

                
            with gr.Column(scale=4):
                source = gr.State('openRouter')
                model = gr.State('openrouter/free')
                temperature = gr.State(1)

                def update_chatbox(model):
                    return mSeries.promptList[model][:-1]

                    
                def reset_text():
                        return ''

                chat_history = gr.Chatbot()    
                response_box = gr.Markdown(label='Response')
                    
                user_input = gr.Textbox(placeholder='Enter your prompt')
                submit= gr.Button('enter',size='sm')

                submit.click(wrapLlm,inputs=[user_input,source,model,temperature], outputs=[response_box]).then(fn=update_chatbox, inputs=[model], outputs=chat_history).then(fn = reset_text, outputs=[user_input])
                user_input.submit(wrapLlm,inputs=[user_input,source,model,temperature ], outputs=[response_box]).then(fn=update_chatbox, inputs=[model], outputs=chat_history).then(fn = reset_text, outputs=[user_input])


            with gr.Column(scale=1):
                with gr.Accordion('Adv_settings'):
                    source_selection = gr.Dropdown(choices=['openRouter','gemini','ollama'],label='Select source-')

                    def get_source(src):
                        return src

                    source_selection.change(fn= get_source, inputs=[source_selection], outputs= [source])

                    temperature_select = gr.Slider(0,100,int,label='temp_slider') 

                    model_name = gr.Textbox(placeholder='Enter model name')
                    
                    def get_model(model_name):
                        return model_name
                    def get_temperature(temperature):
                        return temperature
                    

                    model_name.submit(fn = get_model, inputs=[model_name], outputs=[model]).then(fn = reset_text, outputs=[model_name])
                    temperature.change(fn = get_temperature, inputs=[temperature_select], outputs=[temperature])

            
                files = gr.File(label='insert file',file_count='single',file_types=['pdf'])  



demo.launch()

C:\Users\PGCP-AI\AppData\Local\Temp\ipykernel_19088\2991829124.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\queueing.py", line 785, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\blocks.py", line 2172, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\blocks.py", line 1646, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\utils.py", line

In [13]:
mSeries.promptList

{}

In [14]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    with gr.Row():
        # LEFT COLUMN: Sidebar (Scale 1)
        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("### 💬 Chats")
            new_chat_btn = gr.Button('＋ New Chat', variant='primary', size='sm')
            # Using buttons for a cleaner sidebar look
            gr.Button("History: Project Alpha", size="sm", variant="secondary")
            gr.Button("History: Code Review", size="sm", variant="secondary")

        # MIDDLE COLUMN: Main Chat (Scale 4)
        with gr.Column(scale=4):
            chat_history = gr.Chatbot(height=500)    
            with gr.Group(): # Bonds textbox and button together
                user_input = gr.Textbox(
                    placeholder='Enter your prompt...',
                    show_label=False,
                    container=False
                )
                submit_btn = gr.Button("Send", variant="primary")

        # RIGHT COLUMN: Settings (Scale 1.5)
        with gr.Column(scale=1.5, variant="panel"):
            gr.Markdown("### ⚙️ Configuration")
            source_selection = gr.Dropdown(
                choices=['ollama', 'gemini', 'openRouter'], 
                label='Model Provider', 
                value='ollama'
            )
            
            with gr.Accordion('Model Parameters', open=True):
                temperature = gr.Slider(0, 100, value=70, label='Creativity (Temp)')
                top_p = gr.Slider(0, 1, value=0.9, label='Top P')
            
            files = gr.File(
                label='Reference PDF', 
                file_count='single', 
                file_types=['.pdf']
            )

demo.launch()

C:\Users\PGCP-AI\AppData\Local\Temp\ipykernel_19088\3522000354.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\layouts\column.py:59: UserWarning: 'scale' value should be an integer. Using 1.5 will cause issues.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\helpers.py:1083: UserWarning: Unexpected argument. Filling with None.
  warnings.warn("Unexpected argument. Filling with None.")
